In [1]:
import re
from nltk.corpus import stopwords
from nltk.stem.porter import PorterStemmer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

In [2]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/E-commerce Smart Search & Recommendation Engine

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/MyDrive/E-commerce Smart Search & Recommendation Engine


In [3]:
import shutil
from pathlib import Path

# Define the source path from your Google Drive
source_path = Path('kaggle.json')

# Define the destination path in the Colab system environment
kaggle_dir = Path.home() / '.kaggle'
dest_path = kaggle_dir / 'kaggle.json'

# Create the hidden .kaggle directory if it doesn't exist
kaggle_dir.mkdir(parents=True, exist_ok=True)

# Copy the json file from Drive to the hidden system directory
shutil.copy(source_path, dest_path)

# Set the required permissions (read/write for owner only) so Kaggle accepts it
dest_path.chmod(0o600)

print(f"Kaggle credentials successfully set up at: {dest_path}")

Kaggle credentials successfully set up at: /root/.kaggle/kaggle.json


In [4]:
import zipfile
from pathlib import Path

project_dir = Path('.')
data_dir = project_dir / 'data'
zip_path = project_dir / 'amazon-sales-dataset.zip'


data_dir.mkdir(exist_ok=True)


if not zip_path.exists():
    !kaggle datasets download -d karkavelrajaj/amazon-sales-dataset -p .


with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(data_dir)

print(f"Dataset successfully extracted to: {data_dir.resolve()}")

Dataset successfully extracted to: /content/drive/MyDrive/E-commerce Smart Search & Recommendation Engine/data


In [18]:
import pandas as pd

# Join the directory path with the exact CSV file name
csv_path = data_dir / 'amazon.csv'

# Verify the file exists before loading
if csv_path.exists():
    amazon_df= pd.read_csv(csv_path)

    # Display the first 5 rows to confirm it loaded correctly
    display(amazon_df.head())
else:
    print(f"Error: Could not find the dataset at {csv_path}")

,product_id,product_name,category,discounted_price,actual_price,discount_percentage,rating,rating_count,about_product,user_id,user_name,review_id,review_title,review_content,img_link,product_link
0,B07JW9H4J1,Wayona Nylon Braided USB to Lightning Fast Cha...,Computers&Accessories|Accessories&Peripherals|...,₹399,"₹1,099",64%,4.2,"24,269",High Compatibility : Compatible With iPhone 12...,"AG3D6O4STAQKAY2UVGEUV46KN35Q,AHMY5CWJMMK5BJRBB...","Manav,Adarsh gupta,Sundeep,S.Sayeed Ahmed,jasp...","R3HXWT0LRP0NMF,R2AJM3LFTLZHFO,R6AQJGUP6P86,R1K...","Satisfied,Charging is really fast,Value for mo...",Looks durable Charging is fine tooNo complains...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Wayona-Braided-WN3LG1-Sy...
1,B098NS6PVG,Ambrane Unbreakable 60W / 3A Fast Charging 1.5...,Computers&Accessories|Accessories&Peripherals|...,₹199,₹349,43%,4.0,"43,994","Compatible with all Type C enabled devices, be...","AECPFYFQVRUWC3KGNLJIOREFP5LQ,AGYYVPDD7YG7FYNBX...","ArdKn,Nirbhay kumar,Sagar Viswanathan,Asp,Plac...","RGIQEG07R9HS2,R1SMWZQ86XIN8U,R2J3Y1WL29GWDE,RY...","A Good Braided Cable for Your Type C Device,Go...",I ordered this cable to connect my phone to An...,https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Ambrane-Unbreakable-Char...
2,B096MSW6CT,Sounce Fast Phone Charging Cable & Data Sync U...,Computers&Accessories|Accessories&Peripherals|...,₹199,"₹1,899",90%,3.9,"7,928",【 Fast Charger& Data Sync】-With built-in safet...,"AGU3BBQ2V2DDAMOAKGFAWDDQ6QHA,AESFLDV2PT363T2AQ...","Kunal,Himanshu,viswanath,sai niharka,saqib mal...","R3J3EQQ9TZI5ZJ,R3E7WBGK7ID0KV,RWU79XKQ6I1QF,R2...","Good speed for earlier versions,Good Product,W...","Not quite durable and sturdy,https://m.media-a...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Sounce-iPhone-Charging-C...
3,B08HDJ86NZ,boAt Deuce USB 300 2 in 1 Type-C & Micro USB S...,Computers&Accessories|Accessories&Peripherals|...,₹329,₹699,53%,4.2,"94,363",The boAt Deuce USB 300 2 in 1 cable is compati...,"AEWAZDZZJLQUYVOVGBEUKSLXHQ5A,AG5HTSFRRE6NL3M5S...","Omkar dhale,JD,HEMALATHA,Ajwadh a.,amar singh ...","R3EEUZKKK9J36I,R3HJVYCLYOY554,REDECAZ7AMPQC,R1...","Good product,Good one,Nice,Really nice product...","Good product,long wire,Charges good,Nice,I bou...",https://m.media-amazon.com/images/I/41V5FtEWPk...,https://www.amazon.in/Deuce-300-Resistant-Tang...
4,B08CF3B7N1,Portronics Konnect L 1.2M Fast Charging 3A 8 P...,Computers&Accessories|Accessories&Peripherals|...,₹154,₹399,61%,4.2,"16,905",[CHARGE & SYNC FUNCTION]- This cable comes wit...,"AE3Q6KSUK5P75D5HFYHCRAOLODSA,AFUGIFH5ZAFXRDSZH...","rahuls6099,Swasat Borah,Ajay Wadke,Pranali,RVK...","R1BP4L2HH9TFUP,R16PVJEXKV6QZS,R2UPDB81N66T4P,R...","As good as original,Decent,Good one for second...","Bought this instead of original apple, does th...",https://m.media-amazon.com/images/W/WEBP_40237...,https://www.amazon.in/Portronics-Konnect-POR-1...


In [6]:
amazon_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1465 entries, 0 to 1464
Data columns (total 16 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   product_id           1465 non-null   object
 1   product_name         1465 non-null   object
 2   category             1465 non-null   object
 3   discounted_price     1465 non-null   object
 4   actual_price         1465 non-null   object
 5   discount_percentage  1465 non-null   object
 6   rating               1465 non-null   object
 7   rating_count         1463 non-null   object
 8   about_product        1465 non-null   object
 9   user_id              1465 non-null   object
 10  user_name            1465 non-null   object
 11  review_id            1465 non-null   object
 12  review_title         1465 non-null   object
 13  review_content       1465 non-null   object
 14  img_link             1465 non-null   object
 15  product_link         1465 non-null   object
dtypes: obj

In [7]:
amazon_df.isnull().sum()

,0
product_id,0
product_name,0
category,0
discounted_price,0
actual_price,0
discount_percentage,0
rating,0
rating_count,2
about_product,0
user_id,0


In [8]:
amazon_df.duplicated().sum()

np.int64(0)

In [21]:
amazon_df.duplicated(subset=['product_name','about_product']).sum()

np.int64(118)

In [22]:
amazon_df.drop_duplicates(subset=['product_name','about_product'],inplace=True)

In [24]:
amazon_df.duplicated(subset=['product_name','about_product']).sum()

np.int64(0)

In [25]:
amazon_df['product_content']=amazon_df['product_name']+' '+amazon_df['about_product']

In [26]:
amazon_df.duplicated(subset=['product_content']).sum()

np.int64(0)

In [27]:
amazon_df['product_content'].values

array(["Wayona Nylon Braided USB to Lightning Fast Charging and Data Sync Cable Compatible for iPhone 13, 12,11, X, 8, 7, 6, 5, iPad Air, Pro, Mini (3 FT Pack of 1, Grey) High Compatibility : Compatible With iPhone 12, 11, X/XsMax/Xr ,iPhone 8/8 Plus,iPhone 7/7 Plus,iPhone 6s/6s Plus,iPhone 6/6 Plus,iPhone 5/5s/5c/se,iPad Pro,iPad Air 1/2,iPad mini 1/2/3,iPod nano7,iPod touch and more apple devices.|Fast Charge&Data Sync : It can charge and sync simultaneously at a rapid speed, Compatible with any charging adaptor, multi-port charging station or power bank.|Durability : Durable nylon braided design with premium aluminum housing and toughened nylon fiber wound tightly around the cord lending it superior durability and adding a bit to its flexibility.|High Security Level : It is designed to fully protect your device from damaging excessive current.Copper core thick+Multilayer shielding, Anti-interference, Protective circuit equipment.|WARRANTY: 12 months warranty and friendly customer se

In [28]:
def clean_ecommerce_text(content):
    cleaned = re.sub('[^a-zA-Z0-9]', ' ', str(content))
    cleaned = cleaned.lower()
    cleaned = [word for word in cleaned.split() if word not in stopwords.words('english')]
    return ' '.join(cleaned)

In [29]:
amazon_df['clean_content'] = amazon_df['product_content'].apply(clean_ecommerce_text)

In [30]:
amazon_df['clean_content'].values

array(['wayona nylon braided usb lightning fast charging data sync cable compatible iphone 13 12 11 x 8 7 6 5 ipad air pro mini 3 ft pack 1 grey high compatibility compatible iphone 12 11 x xsmax xr iphone 8 8 plus iphone 7 7 plus iphone 6s 6s plus iphone 6 6 plus iphone 5 5s 5c se ipad pro ipad air 1 2 ipad mini 1 2 3 ipod nano7 ipod touch apple devices fast charge data sync charge sync simultaneously rapid speed compatible charging adaptor multi port charging station power bank durability durable nylon braided design premium aluminum housing toughened nylon fiber wound tightly around cord lending superior durability adding bit flexibility high security level designed fully protect device damaging excessive current copper core thick multilayer shielding anti interference protective circuit equipment warranty 12 months warranty friendly customer services ensures long time enjoyment purchase meet question problem please hesitate contact us',
       'ambrane unbreakable 60w 3a fast charg

In [63]:
vectorizer = TfidfVectorizer(max_df=90, min_df=2)
tfidf_matrix = vectorizer.fit_transform(amazon_df['clean_content'])

In [64]:
tfidf_matrix.shape

(1347, 5760)

In [65]:
def search_products(query: str, top_n: int = 5) -> None:
    """
    Search for products using a text query.
    """
    # 1. Convert the user query into numbers (vector) using our vectorizer.
    # It changes the text into a row of 10190 numbers.
    query_vec = vectorizer.transform([query])

    # 2. Calculate the similarity between the user query and all products.
    # .flatten() makes the result a simple 1D list.
    similarities = cosine_similarity(query_vec, tfidf_matrix).flatten()

    # 3. Sort products from highest to lowest similarity and get the best results.
    top_indices = similarities.argsort()[-top_n:][::-1]

    print(f"Search results for: '{query}'\n" + "="*50)

    # 4. Print the results.
    for idx in top_indices:
        # Check if there is a real match (similarity is greater than 0).
        if similarities[idx] > 0:
            print(f"Similarity Score: {similarities[idx]:.2f}")
            # Print the product name from the dataframe.
            print(f"Product: {amazon_df['product_name'].iloc[idx]}")
            print("-" * 50)
        else:
            print("No matching products found.")
            break

# Test the search function
search_products("fast charging cable iphone 13")

Search results for: 'fast charging cable iphone 13'
Similarity Score: 0.69
Product: REDTECH USB-C to Lightning Cable 3.3FT, [Apple MFi Certified] Lightning to Type C Fast Charging Cord Compatible with iPhone 14/13/13 pro/Max/12/11/X/XS/XR/8, Supports Power Delivery - White
--------------------------------------------------
Similarity Score: 0.59
Product: OpenTech® Military-Grade Tempered Glass Screen Protector Compatible for iPhone 13/13 Pro / 14 with Edge to Edge Coverage and Easy Installation kit (6.1 Inches)
--------------------------------------------------
Similarity Score: 0.55
Product: POPIO Tempered Glass Compatible for iPhone 13 / iPhone 13 Pro/iPhone 14 (Transparent) Edge to Edge Full Screen Coverage with Installation Kit, Pack of 2
--------------------------------------------------
Similarity Score: 0.49
Product: ESR USB C to Lightning Cable, 10 ft (3 m), MFi-Certified, Braided Nylon Power Delivery Fast Charging for iPhone 14/14 Plus/14 Pro/14 Pro Max, iPhone 13/12/11/X/8 Se

In [68]:
def recommend_products(product_index: int, top_n: int = 5) -> None:
    """
    Recommend similar products based on a chosen product.
    """
    # 1. Get the vector of the selected product from our matrix.
    # This represents the features of the product the user is viewing.
    target_vec = tfidf_matrix[product_index]

    # 2. Calculate the similarity between this product and all other products.
    similarities = cosine_similarity(target_vec, tfidf_matrix).flatten()

    # 3. Sort the results. We take top_n + 1 because the product itself will be a 100% match.
    # We use [1:] to skip the first result (the exact same product).
    top_indices = similarities.argsort()[-(top_n + 1):][::-1][1:]

    # Get the name of the target product for the print statement.
    target_name = amazon_df['product_name'].iloc[product_index]
    print(f"Because you viewed: '{target_name[:50]}...'\nWe recommend:\n" + "="*50)

    # 4. Print the recommended products.
    for idx in top_indices:
        print(f"Similarity Score: {similarities[idx]:.2f}")
        print(f"Product: {amazon_df['product_name'].iloc[idx]}")
        print("-" * 50)

# Test the recommendation function using product at index 0
recommend_products(0)

Because you viewed: 'Wayona Nylon Braided USB to Lightning Fast Chargin...'
We recommend:
Similarity Score: 0.97
Product: Wayona Nylon Braided Usb Syncing And Charging Cable Sync And Charging Cable For Iphone, Ipad (3 Ft, Black) - Pack Of 2
--------------------------------------------------
Similarity Score: 0.95
Product: Wayona Nylon Braided (2 Pack) Lightning Fast Usb Data Cable Fast Charger Cord For Iphone, Ipad Tablet (3 Ft Pack Of 2, Grey)
--------------------------------------------------
Similarity Score: 0.95
Product: Wayona Nylon Braided 3A Lightning to USB A Syncing and Fast Charging Data Cable for iPhone, Ipad (3 FT Pack of 1, Black)
--------------------------------------------------
Similarity Score: 0.75
Product: Wayona Usb Nylon Braided Data Sync And Charging Cable For Iphone, Ipad Tablet (Red, Black)
--------------------------------------------------
Similarity Score: 0.74
Product: Wayona Nylon Braided 2M / 6Ft Fast Charge Usb To Lightning Data Sync And Charging Cable Fo

In [70]:
project_dir = Path('.')

In [69]:
!pip install -q streamlit
!npm install localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 15.8 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
added 22 packages in 5s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴

In [77]:
import streamlit as st
import os
import sys
current_dir = os.getcwd()
sys.path.append(current_dir)

In [82]:
# 1. Run Streamlit in the background with correct nested quotes around the path
get_ipython().system_raw("streamlit run '/content/drive/MyDrive/E-commerce Smart Search & Recommendation Engine/E_app.py' --server.port 8501 --server.address 0.0.0.0 &")

# 2. Wait a few seconds to ensure Streamlit starts
import time
time.sleep(3)

# 3. Start Cloudflare tunnel
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
!chmod +x cloudflared-linux-amd64
!./cloudflared-linux-amd64 tunnel --url http://localhost:8501

2026-10-20T09:24:36Z INF Thank you for trying Cloudflare Tunnel. Doing so, without a Cloudflare account, is a quick way to experiment and try it out. However, be aware that these account-less Tunnels have no uptime guarantee, are subject to the Cloudflare Online Services Terms of Use (https://www.cloudflare.com/website-terms/), and Cloudflare reserves the right to investigate your use of Tunnels for violations of such terms. If you intend to use Tunnels in production you should use a pre-created named tunnel by following: https://developers.cloudflare.com/cloudflare-one/connections/connect-apps
2026-10-20T09:24:36Z INF Requesting new quick Tunnel on trycloudflare.com...
2026-10-20T09:24:39Z INF +--------------------------------------------------------------------------------------------+
2026-10-20T09:24:39Z INF |  Your quick Tunnel has been created! Visit it at (it may take some time to be reachable):  |
2026-10-20T09:24:39Z INF |  https://howard-departure-vacations-hawaiian.trycloudf

In [84]:
pip install pipreqs

  Preparing metadata (setup.py) ... done
  Created wheel for docopt: filename=docopt-0.6.2-py2.py3-none-any.whl size=13781 sha256=672d49ba30babe0b2b47e6b898341b6f97b8e6aba06114fab5bb68533d684080
  Stored in directory: /root/.cache/pip/wheels/0b/1d/03/175286677fb5a1341cc3e4755bf8ec0ed08f3329afd67446b0
Successfully built docopt


In [86]:
!pipreqs . --force

Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
Please, verify manually the final list of requirements.txt to avoid possible dependency confusions.
INFO: Successfully saved requirements file in ./requirements.txt
